In [0]:
# The purpose of this notebook is to initialize two lists. The first is a list of stations
# where we will update forecasts from and the second is a mapping from the citibike locations
# to the nearest station. We will update the list of stations when the station_info table is updated
# and new station locations are discovered.


# Here is information on the two files

# hrrr_locations.json
# model: weather model to request.
# grid_options: settings shared by mapping and forecast downloads.
# locations: one entry per weather cell:
#   location_id: identifier based on the selected grid-cell coordinates.
#   latitude, longitude: representative coordinates sent to the API.
#
# station_weather_lookup.csv
# station_id: Citi Bike station identifier.
# lat, lon: observed Citi Bike station coordinates.
# location_id: assigned weather cell, matching an entry in the JSON.

# Our usual imports. We will just use pandas instead of spark like the majority
# of our other notebooks because this is a small list

from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import time
import pandas as pd

# Standard boilerplate where we define important constants and the key paths

# Define the source table, output directory, and weather API settings.

SOURCE_TABLE = "citibike_project.citibike.silver_station_info"

# Just a note that this path determines if you are in _dev or prod

REFERENCE_DIRECTORY = Path(
    "/Volumes/citibike_project/citibike/raw/weather_forecast/_dev/reference"
)

# Prod version
#REFERENCE_DIRECTORY = Path(
#    "/Volumes/citibike_project/citibike/raw/weather_forecast/reference"
#)

# For now we will use open-meteo.com for weather forecasts
# It is debatable whether at some point we should just use the national weather
# service.

# The gfs_hrrr weather forecast model is a high-resolution rapid refresh weather model, see
# https://open-meteo.com/en/docs/gfs-api

API_URL = "https://api.open-meteo.com/v1/forecast"
MODEL = "gfs_hrrr"

# cell_selection:land picks closest land station when we search
# The elevation:nan is probably not needed since NYC is so flat, but
# it gives a forecast at the mean elevation of the grid cell. This also
# allows the station finding to use just horizontal distance

GRID_OPTIONS = {
    "cell_selection": "land",
    "elevation": "nan",
}

# Collect each distinct station and coordinate combination from silver.
# We will use these to identify the closest sation
station_locations = (
    spark.table(SOURCE_TABLE)
    .select("station_id", "lat", "lon")
    .distinct()
    .toPandas()
)

# Query the locations
request_locations = (
    station_locations[["lat", "lon"]]
    .drop_duplicates()
    .sort_values(["lat", "lon"])
    .reset_index(drop=True)
)

# Accumulate the weather-cell assignment returned for each coordinate pair.
grid_assignments = []

# Query 50 locations at a time to reduce the number of HTTP requests.
for batch_start in range(0, len(request_locations), 50):
    location_batch = request_locations.iloc[batch_start:batch_start + 50]

    # Even though we aren't asking for any specific forecast the API should still
    # return the location data/info

    request_parameters = {
        "latitude": ",".join(location_batch["lat"].astype(str)),
        "longitude": ",".join(location_batch["lon"].astype(str)),
        "models": MODEL,
        **GRID_OPTIONS,
        "elevation": ",".join(["nan"] * len(location_batch)),
    }

    # Send the request and decode the JSON response.
    with urlopen(
        f"{API_URL}?{urlencode(request_parameters)}", timeout=60
        ) as response:
            forecasts = json.load(response)

        # Wrap a single-location response so the following loop always receives a list.
        # Prevents a problem if the number of stations is almost a multiple of 50
    if isinstance(forecasts, dict):
        forecasts = [forecasts]

        # Pair each requested location with its returned cell and assign a consistent ID.
    for location, forecast in zip(
        location_batch.itertuples(index=False), forecasts, strict=True
        ):
        grid_assignments.append({
            "lat": location.lat,
            "lon": location.lon,
            "location_id": (
                f"hrrr_{forecast['latitude']:.6f}_"
                f"{forecast['longitude']:.6f}"
            ),
        })

    # Report progress and pause to maintain polite API rate policy
    print(f"Mapped {len(grid_assignments):,}/{len(request_locations):,}")
    time.sleep(6)

# Convert the accumulated assignments into a table.
grid_assignments = pd.DataFrame(grid_assignments)

# Attach the weather-cell assignment to every station at those coordinates.
station_weather_lookup = (
    station_locations
    .merge(grid_assignments, on=["lat", "lon"], validate="many_to_one")
    .sort_values(["station_id", "lat", "lon"])
)

# Keep one representative request location per cell for the downloader.
weather_locations = (
    grid_assignments
    .drop_duplicates("location_id")
    .rename(columns={"lat": "latitude", "lon": "longitude"})
    [["location_id", "latitude", "longitude"]]
    .sort_values("location_id")
)

# Package the locations with the settings needed to reproduce their cell selection.
weather_configuration = {
    "model": MODEL,
    "grid_options": GRID_OPTIONS,
    "locations": weather_locations.to_dict(orient="records"),
}

# Create the output directory and write both reference files.
REFERENCE_DIRECTORY.mkdir(parents=True, exist_ok=True)

(REFERENCE_DIRECTORY / "hrrr_locations.json").write_text(
    json.dumps(weather_configuration, indent=2), encoding="utf-8"
)
station_weather_lookup.to_csv(
    REFERENCE_DIRECTORY / "station_weather_lookup.csv", index=False
)

print(f"Saved {len(weather_locations):,} weather locations.")
print(f"Saved {len(station_weather_lookup):,} station/location mappings.")
print(f"Directory: {REFERENCE_DIRECTORY}")